In [1]:
import pandas as pd
import os

In [4]:
!find / -path "*strategy_b*/*.csv" 2>/dev/null

/data/processed/strategy_b/train.csv
/data/processed/strategy_b/validation.csv
/data/processed/strategy_b/test.csv


In [8]:
paths = {
    "train": "/data/processed/strategy_b/train.csv",
    "validation": "/data/processed/strategy_b/validation.csv",
    "test": "/data/processed/strategy_b/test.csv",
}

for split_name, path in paths.items():
    print(split_name, "->", os.path.exists(path))

train -> True
validation -> True
test -> True


In [9]:
dfs = {}

for split_name, path in paths.items():
    dfs[split_name] = pd.read_csv(path)
    print("\n" + "=" * 40)
    print(split_name)
    print("Shape:", dfs[split_name].shape)
    print("Columns:", dfs[split_name].columns.tolist())
    print("Label distribution:")
    print(dfs[split_name]["label"].value_counts().sort_index())


train
Shape: (11575, 8)
Columns: ['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']
Label distribution:
label
0    7085
1    4490
Name: count, dtype: int64

validation
Shape: (1447, 8)
Columns: ['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']
Label distribution:
label
0    886
1    561
Name: count, dtype: int64

test
Shape: (1447, 8)
Columns: ['input_text', 'label', 'Score', 'soru', 'context', 'cevap', 'kaynak', 'veri türü']
Label distribution:
label
0    885
1    562
Name: count, dtype: int64


In [10]:
majority_class = dfs["train"]["label"].value_counts().idxmax()

print("Majority class:", majority_class)
print("Train label distribution:")
print(dfs["train"]["label"].value_counts())

Majority class: 0
Train label distribution:
label
0    7085
1    4490
Name: count, dtype: int64


In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

y_test = dfs["test"]["label"]
y_pred_majority = np.full(shape=len(y_test), fill_value=majority_class)

baseline_accuracy = accuracy_score(y_test, y_pred_majority)
baseline_macro_f1 = f1_score(y_test, y_pred_majority, average="macro")
baseline_weighted_f1 = f1_score(y_test, y_pred_majority, average="weighted")

print("Majority Class Baseline Results")
print("Accuracy:", baseline_accuracy)
print("Macro F1:", baseline_macro_f1)
print("Weighted F1:", baseline_weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_majority))

Majority Class Baseline Results
Accuracy: 0.6116102280580511
Macro F1: 0.3795025728987993
Weighted F1: 0.4642153103185036

Classification Report:
              precision    recall  f1-score   support

           0       0.61      1.00      0.76       885
           1       0.00      0.00      0.00       562

    accuracy                           0.61      1447
   macro avg       0.31      0.50      0.38      1447
weighted avg       0.37      0.61      0.46      1447



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

y_test = dfs["test"]["label"]
y_pred_majority = np.full(shape=len(y_test), fill_value=majority_class)

baseline_accuracy = accuracy_score(y_test, y_pred_majority)
baseline_macro_f1 = f1_score(y_test, y_pred_majority, average="macro", zero_division=0)
baseline_weighted_f1 = f1_score(y_test, y_pred_majority, average="weighted", zero_division=0)

print("Majority Class Baseline Results")
print("Accuracy:", baseline_accuracy)
print("Macro F1:", baseline_macro_f1)
print("Weighted F1:", baseline_weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_majority, zero_division=0))

Majority Class Baseline Results
Accuracy: 0.6116102280580511
Macro F1: 0.3795025728987993
Weighted F1: 0.4642153103185036

Classification Report:
              precision    recall  f1-score   support

           0       0.61      1.00      0.76       885
           1       0.00      0.00      0.00       562

    accuracy                           0.61      1447
   macro avg       0.31      0.50      0.38      1447
weighted avg       0.37      0.61      0.46      1447



In [13]:
import os

os.makedirs("/outputs/tables", exist_ok=True)

baseline_results = pd.DataFrame([
    {
        "strategy": "strategy_b",
        "baseline_type": "majority_class",
        "majority_class": majority_class,
        "test_accuracy": baseline_accuracy,
        "test_macro_f1": baseline_macro_f1,
        "test_weighted_f1": baseline_weighted_f1,
        "test_size": len(y_test),
        "label_0_support": int((y_test == 0).sum()),
        "label_1_support": int((y_test == 1).sum())
    }
])

baseline_results

,strategy,baseline_type,majority_class,test_accuracy,test_macro_f1,test_weighted_f1,test_size,label_0_support,label_1_support
0,strategy_b,majority_class,0,0.61161,0.379503,0.464215,1447,885,562


In [14]:
baseline_results.to_csv("/outputs/tables/majority_baseline_strategy_b.csv", index=False)

print("Baseline results saved.")
print("/outputs/tables/majority_baseline_strategy_b.csv")

Baseline results saved.
/outputs/tables/majority_baseline_strategy_b.csv


In [15]:
import os

print(os.path.exists("/outputs/tables/majority_baseline_strategy_b.csv"))

True
